# RemoGen Studio - conversational narration on Colab (ChatTTS)

Runs ChatTTS here on Colab's GPU. It's trained specifically on natural dialogue, so narration
comes out sounding less like a teleprompter reading and more like a person talking - it adds
natural pauses and can include `[laugh]`/`[uv_break]`-style delivery.

**Read this before using it: ChatTTS does NOT clone your uploaded voice sample.** There is no
zero-shot "clone this audio" feature in ChatTTS the way there is in XTTS or Dia. What you get
instead is ONE consistent, natural-sounding voice, randomly selected once per project, that
stays the same across every scene - uploading a reference sample just gives that consistent
voice a stable ID to remember, it does not change what the voice sounds like.

**How to use**
1. **Runtime -> Change runtime type -> T4 GPU**.
2. Run the cells below **in order**.
3. The last cell prints a **URL** and an **API key** - paste both into RemoGen Studio -> **Voice Engine**, **Test connection**, **Save**.
4. Leave this tab open while you generate. Re-running the last cell gives a **new** URL and key every time.

In [ ]:
# 1) Check for a GPU and install the dependencies (first run: ~2-3 minutes)
!nvidia-smi -L || echo ">>> NO GPU: choose Runtime -> Change runtime type -> T4 GPU, then re-run"
!pip -q install ChatTTS fastapi "uvicorn[standard]" python-multipart

In [ ]:
%%writefile rg_common.py
"""
Shared plumbing for RemoGen's Colab TTS servers (server_xtts.py, server_dia.py,
server_chattts.py). All three expose the SAME contract:
  GET  /health   -> {"status": "ok" | "loading", "model", "device", "gpu"}
  POST /voices   -> multipart "file" -> {"voice_id": "..."}
  POST /tts      -> JSON {"text", "language", "speed", "voice_id"} -> audio/wav
so the Node backend (services/tts/colabProvider.js) never needs to know or
care which model is actually running - only this file's contents differ.
"""
import glob
import hashlib
import hmac
import io
import math
import os
import struct
import subprocess
import wave
from typing import Optional

from fastapi import Header, HTTPException

API_KEY = os.environ.get("RG_API_KEY", "")
FAKE = os.environ.get("RG_FAKE_TTS") == "1"
VOICE_DIR = os.environ.get("RG_VOICE_DIR", "/tmp/rg_voices")
os.makedirs(VOICE_DIR, exist_ok=True)


def require_key(x_api_key: str = Header(default="")):
    if API_KEY and not hmac.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Invalid API key")


def voice_path(voice_id: str) -> Optional[str]:
    """Resolves a voice_id to its uploaded file, refusing anything that isn't
    a plain hex id - blocks path-traversal tricks like "../../etc/passwd"."""
    if not voice_id or not voice_id.isalnum():
        return None
    matches = glob.glob(os.path.join(VOICE_DIR, f"{voice_id}.*"))
    return matches[0] if matches else None


def save_uploaded_voice(data: bytes, original_filename: str) -> str:
    """Saves an uploaded reference sample, deduped by content hash so
    re-uploading the same file is free, and normalises it to mono 22.05kHz
    (what these models expect) when ffmpeg is available."""
    if not data:
        raise HTTPException(status_code=400, detail="Empty file")
    if len(data) > 50 * 1024 * 1024:
        raise HTTPException(status_code=413, detail="Reference audio is over 50 MB")

    voice_id = hashlib.sha1(data).hexdigest()[:16]
    ext = os.path.splitext(original_filename or "")[1].lower() or ".wav"
    raw_path = os.path.join(VOICE_DIR, f"{voice_id}{ext}")
    with open(raw_path, "wb") as fh:
        fh.write(data)

    wav_path = os.path.join(VOICE_DIR, f"{voice_id}.wav")
    if raw_path != wav_path:
        try:
            subprocess.run(
                ["ffmpeg", "-y", "-loglevel", "error", "-i", raw_path, "-ac", "1", "-ar", "22050", wav_path],
                check=True, timeout=120,
            )
            os.remove(raw_path)
        except Exception:
            pass  # keep the original file; most loaders can read it directly
    return voice_id


def wav_duration_seconds(path: str) -> float:
    with wave.open(path, "rb") as w:
        return w.getnframes() / w.getframerate()


def apply_speed(input_path: str, output_path: str, speed: float):
    """Re-times narration with ffmpeg's atempo filter. Used uniformly across
    all three engines rather than each model's own (inconsistent, sometimes
    absent) speed control, so behaviour matches what services/tts/http.js's
    caller expects regardless of which Colab engine is running."""
    speed = max(0.5, min(2.0, float(speed or 1.0)))
    if abs(speed - 1.0) < 0.01:
        if input_path != output_path:
            os.replace(input_path, output_path)
        return
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", input_path, "-filter:a", f"atempo={speed}", output_path],
        check=True, timeout=120,
    )


def fake_wav_bytes(text: str) -> bytes:
    """A short sine-wave tone standing in for real audio, so the HTTP
    plumbing (auth, upload, download) can be tested without a GPU or any of
    these (large) models actually installed. Set RG_FAKE_TTS=1 to use this."""
    seconds = max(0.6, 0.4 * len(text.split()))
    rate = 24000
    frames = b"".join(
        struct.pack("<h", int(6000 * math.sin(2 * math.pi * 220 * i / rate))) for i in range(int(seconds * rate))
    )
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes(frames)
    return buf.getvalue()


In [ ]:
%%writefile server_chattts.py
"""
RemoGen Studio - ChatTTS voice server for Google Colab.
See rg_common.py for the shared HTTP contract all three engine servers share.

IMPORTANT, read this before assuming this clones your voice - IT DOES NOT.
ChatTTS has no zero-shot "clone this uploaded audio" capability the way
XTTS/Dia do. What it offers instead is a persistent RANDOM speaker voice:
`sample_random_speaker()` returns a fixed embedding you can reuse so every
scene in a project sounds like the same (randomly chosen, natural-sounding)
narrator - but it is NOT derived from whatever reference sample the user
uploaded. This file is honest about that: an uploaded sample's CONTENT is
only used as a key (so the same upload consistently maps to the same
sampled voice across retries), never as the source of the voice itself.
The frontend/README must not claim this clones the user's voice.

What ChatTTS is genuinely good at: natural conversational fillers via
[uv_break]/[laugh]-style tags and a statistically natural, less "reading
a teleprompter" delivery than a straight narration model.
"""
import json
import os
import threading
from contextlib import asynccontextmanager
from typing import Optional

from fastapi import Depends, FastAPI, File, HTTPException, UploadFile
from fastapi.responses import JSONResponse, Response
from pydantic import BaseModel

from rg_common import FAKE, VOICE_DIR, require_key, save_uploaded_voice, apply_speed, fake_wav_bytes

# How much natural "oral" filler / laughter / pausing the refine-text stage
# adds - see ChatTTS's own docs for the [oral_N][laugh_N][break_N] scale.
REFINE_PROMPT = os.environ.get("CHATTTS_REFINE_PROMPT", "[oral_2][laugh_0][break_4]")

state = {"chat": None, "device": "cpu", "gpu": None, "default_speaker": None, "error": None}
synth_lock = threading.Lock()

SPEAKER_STORE = os.path.join(VOICE_DIR, "chattts_speakers.json")
speakers = {}  # content-hash voice_id -> saved speaker embedding string
if os.path.exists(SPEAKER_STORE):
    try:
        speakers.update(json.load(open(SPEAKER_STORE)))
    except Exception:
        pass


def save_speakers():
    json.dump(speakers, open(SPEAKER_STORE, "w"))


def load_model():
    try:
        if FAKE:
            state.update(device="fake", gpu="none (fake mode)", default_speaker="fake-speaker", chat="fake")
            return
        import torch
        import ChatTTS

        device = "cuda" if torch.cuda.is_available() else "cpu"
        state["device"] = device
        state["gpu"] = torch.cuda.get_device_name(0) if device == "cuda" else None

        chat = ChatTTS.Chat()
        chat.load_models(compile=False)
        state["chat"] = chat
        # One consistent voice for projects that don't upload a reference
        # sample at all, sampled once per Colab session so it doesn't
        # change scene-to-scene.
        state["default_speaker"] = str(chat.sample_random_speaker())
    except Exception as exc:
        state["error"] = f"{type(exc).__name__}: {exc}"
        raise


@asynccontextmanager
async def lifespan(_app):
    threading.Thread(target=load_model, daemon=True).start()
    yield


app = FastAPI(title="RemoGen Colab TTS - ChatTTS", lifespan=lifespan)


@app.get("/health", dependencies=[Depends(require_key)])
def health():
    if state["error"]:
        return JSONResponse(status_code=500, content={"status": "error", "detail": state["error"]})
    ready = state["chat"] is not None
    return {
        "status": "ok" if ready else "loading",
        "model": "chattts" if not FAKE else "fake",
        "device": state["device"],
        "gpu": state["gpu"],
        # Surfaced so the app can be upfront in its UI that this isn't cloning.
        "clones_voice": False,
    }


@app.post("/voices", dependencies=[Depends(require_key)])
async def upload_voice(file: UploadFile = File(...)):
    data = await file.read()
    # save_uploaded_voice() is only used here to get a stable content-hash
    # id and to validate the upload - the audio ITSELF is discarded after,
    # since ChatTTS has nothing that can use it.
    voice_id = save_uploaded_voice(data, file.filename)

    if voice_id not in speakers:
        if FAKE:
            speakers[voice_id] = "fake-sampled-speaker"
        else:
            if state["chat"] is None:
                raise HTTPException(status_code=503, detail="Model is still loading - try again shortly")
            speakers[voice_id] = str(state["chat"].sample_random_speaker())
        save_speakers()

    return {"voice_id": voice_id}


class TTSRequest(BaseModel):
    text: str
    language: str = "en"  # accepted for interface consistency; ChatTTS handles mixed-language text natively
    speed: float = 1.0
    voice_id: Optional[str] = None


@app.post("/tts", dependencies=[Depends(require_key)])
def synthesize(req: TTSRequest):
    text = req.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="text is empty")
    if state["error"]:
        raise HTTPException(status_code=500, detail=state["error"])
    if state["chat"] is None and not FAKE:
        raise HTTPException(status_code=503, detail="Model is still loading")

    speaker_emb = speakers.get(req.voice_id) if req.voice_id else None
    if req.voice_id and speaker_emb is None:
        raise HTTPException(status_code=404, detail="Unknown voice_id - upload the reference sample again")
    speaker_emb = speaker_emb or state["default_speaker"]

    raw_path = os.path.join(VOICE_DIR, f"chattts_raw_{threading.get_ident()}_{os.getpid()}.wav")
    final_path = os.path.join(VOICE_DIR, f"chattts_out_{threading.get_ident()}_{os.getpid()}.wav")

    if FAKE:
        with open(raw_path, "wb") as fh:
            fh.write(fake_wav_bytes(text))
    else:
        import torch
        import torchaudio
        import ChatTTS

        with synth_lock:
            params_infer_code = ChatTTS.Chat.InferCodeParams(spk_emb=speaker_emb)
            params_refine_text = ChatTTS.Chat.RefineTextParams(prompt=REFINE_PROMPT)
            wavs = state["chat"].infer([text], params_infer_code=params_infer_code, params_refine_text=params_refine_text)
        torchaudio.save(raw_path, torch.from_numpy(wavs[0]), 24000)

    apply_speed(raw_path, final_path, req.speed)
    with open(final_path, "rb") as fh:
        audio = fh.read()
    for p in (raw_path, final_path):
        if os.path.exists(p):
            os.remove(p)
    return Response(content=audio, media_type="audio/wav")


In [ ]:
# Start the TTS server + a public tunnel, then print the URL and API key
import os, re, sys, time, json, secrets, subprocess, urllib.request, urllib.error

os.environ["COQUI_TOS_AGREED"] = "1"     # harmless if this engine doesn't use it
API_KEY = secrets.token_urlsafe(24)      # protects the public URL - don't share it
PORT = 8000

def health():
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/health", headers={"X-API-Key": API_KEY})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())

# --- start the server (loads the model in the background) ---
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server_chattts:app", "--host", "127.0.0.1", "--port", str(PORT)],
    env={**os.environ, "RG_API_KEY": API_KEY},
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
print("Loading ChatTTS (first run downloads the model, a couple of minutes)...")
t0 = time.time()
while True:
    if server.poll() is not None:
        print(open("server.log").read()[-3000:])
        raise SystemExit("Server crashed - see the log above")
    try:
        info = health()
        if info.get("status") == "ok":
            break
    except urllib.error.HTTPError as e:
        if e.code == 500:
            raise SystemExit("Model failed to load: " + e.read().decode()[:1500])
    except Exception:
        pass                                   # server still starting
    if time.time() - t0 > 600:
        raise SystemExit("Timed out waiting for the model to load")
    time.sleep(5)
print("Model ready on:", info.get("gpu") or info.get("device"))

# --- open a public tunnel (Cloudflare quick tunnel: free, no account) ---
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "cloudflared"], check=True)
    os.chmod("cloudflared", 0o755)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        public_url = m.group(0)
        break
if not public_url:
    print(open("tunnel.log").read()[-2000:])
    raise SystemExit("Could not open a tunnel - see the log above")

print("\n" + "=" * 64)
print(" PASTE THESE INTO RemoGen Studio -> Voice Engine")
print("=" * 64)
print(f" Engine    : ChatTTS")
print(f" Colab URL : {public_url}")
print(f" API key   : {API_KEY}")
print("=" * 64)
print(" Keep this cell running. Stop it (or close the tab) to shut TTS down.\n")

# --- keep the runtime alive while you work ---
try:
    while server.poll() is None and tunnel.poll() is None:
        time.sleep(30)
except KeyboardInterrupt:
    pass
finally:
    tunnel.terminate(); server.terminate()
    print("Stopped.")

### Troubleshooting
- **"NO GPU"** - Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart and run all.
- **Voice doesn't match my uploaded sample at all** - that's expected, see the note above; ChatTTS picks its own consistent voice rather than cloning yours.
- **Want more/less laughter or pausing** - set the `CHATTTS_REFINE_PROMPT` environment variable before starting the server (see `server_chattts.py`) - e.g. `[oral_2][laugh_1][break_4]` for a bit more laughter.
- **RemoGen says the tunnel is down** - Colab stopped (free tier disconnects after idle/limits). Re-run the last cell and paste the new URL + key.